In [ ]:
# %pip install -q pandas numpy transformers underthesea matplotlib seaborn scikit-learn sentence_transformers

In [ ]:
import time
import pandas as pd
import numpy as np
from underthesea import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
import torch
import gc
gc.collect()
torch.cuda.empty_cache()

# Function definitions

In [ ]:
def get_approx_token_length(chunk:dict) -> int:
    return sum(len(sent.split()) for sent in chunk['sents'])

def greedy_zscore_split(sentences, embeddings, z_threshold=-1.0, max_length=200):
    # 1. Tính tất cả các điểm similarity kế tiếp (Local Cohesion)
    # So sánh câu i với câu i-1 thường phản ánh tính mạch lạc tốt hơn Centroid
    if len(sentences) <= 1:
        return [{'sents': sentences, 'vectors': embeddings, 'score': 0.0}]

    raw_sims = []
    for i in range(1, len(embeddings)):
        s = cosine_similarity([embeddings[i-1]], [embeddings[i]])[0][0]
        raw_sims.append(s)
    
    # 2. Tính Z-score cho toàn bộ các điểm similarity
    avg = np.mean(raw_sims)
    std = np.std(raw_sims)
    z_scores = [(s - avg) / (std + 1e-8) for s in raw_sims]

    # 3. Tiến hành phân nhóm (Greedy với Z-score)
    chunks = []
    current_chunk = {'sents': [sentences[0]], 'vectors': [embeddings[0]]}
    
    for i in range(len(z_scores)):
        sent = sentences[i+1]
        vec = embeddings[i+1]
        z = z_scores[i]
        
        # Kiểm tra điều kiện cắt: Z-score quá thấp (điểm rơi sâu) hoặc quá dài
        is_too_long = get_approx_token_length(current_chunk) > max_length
        
        # Nếu Z-score > z_threshold (ví dụ -1.0), nghĩa là độ tương đồng vẫn nằm trong mức chấp nhận được
        if z >= z_threshold and not is_too_long:
            current_chunk['sents'].append(sent)
            current_chunk['vectors'].append(vec)
            current_chunk['score'] = z # Lưu lại Z-score để debug
        else:
            chunks.append(current_chunk)
            current_chunk = {'sents': [sent], 'vectors': [vec]}

    chunks.append(current_chunk)
    return chunks

def greedy_adaptive_split(sentences:list[str], embeddings:np.ndarray, base_threshold:float=0.6, penalty_alpha=0.05, max_length=200):
    """
    Args:
        - sentences: List of sentences to be segmented.
        - embeddings: Corresponding sentence embeddings as a 2D numpy array.
        - base_threshold: Base cosine similarity threshold for merging sentences.
        - penalty_alpha: Coefficient for the logarithmic penalty based on chunk size.
        - max_length: Maximum approximate token length for each chunk.
    Returns:
        - List of chunks, where each chunk is a dictionary with keys:
            - 'sents': List of sentences in the chunk.
            - 'vectors': List of embeddings in the chunk.
            - 'score': Final similarity score used for the last merge.
    """
    
    chunks = []
    current_chunk = {'sents': [sentences[0]], 'vectors': [embeddings[0]]} # initialize with the first sentence and embedding
    for i, (sent, vec) in enumerate(zip(sentences[1:], embeddings[1:])):
        centroid = np.mean(current_chunk['vectors'], axis=0)
        
        # embeddings now use SentenceTransformer
        sim = cosine_similarity([centroid], [vec])[0][0]
        curr_sent_count = len(current_chunk['sents'])
        
        adaptive_threshold = base_threshold + (penalty_alpha * np.log(curr_sent_count))
        adaptive_threshold = min(adaptive_threshold, 0.90)
        
        is_too_long = get_approx_token_length(current_chunk) > max_length
        
        if sim >= adaptive_threshold and not is_too_long:
            current_chunk['sents'].append(sent)
            current_chunk['vectors'].append(vec)
            current_chunk['score'] = sim
        else:
            chunks.append(current_chunk)
            current_chunk = {'sents': [sent], 'vectors': [vec]}
        
    if current_chunk['sents']:
        chunks.append(current_chunk)
        
    return chunks

In [ ]:
def similarity_based_merge(chunks:list[dict], model:SentenceTransformer, min_syllable_count:int=50, min_sentence_count:int=1) -> list[dict]:
    """
    Args:
        - chunks: List of chunk dictionaries with keys 'sents' and 'vectors'.
        - min_syllable_count: Minimum syllable count below which chunks are considered for merging.
    Returns:
        - Merged list of chunk dictionaries.
    """
    # edge case: first chunk is small, last chunk is small
    i = 0
    while i < len(chunks):
        chunk = chunks[i]
        syllable_count = get_approx_token_length(chunk)
        if syllable_count < min_syllable_count or len(chunk['sents']) <= min_sentence_count:
            curr_chunk_sents = ' '.join(chunk['sents'])
            curr_chunk_embeddings = model.encode([curr_chunk_sents], normalize_embeddings=True)[0]
            
            options = []
            if i > 0:
                prev_chunk = chunks[i-1]
                prev_chunk_sents = ' '.join(prev_chunk['sents'])
                prev_chunk_embeddings = model.encode([prev_chunk_sents], normalize_embeddings=True)[0]
                sim_prev = cosine_similarity([curr_chunk_embeddings], [prev_chunk_embeddings])[0][0]
                options.append(('prev', sim_prev))
            if i < len(chunks) - 1:
                next_chunk = chunks[i+1]
                next_chunk_sents = ' '.join(next_chunk['sents'])
                next_chunk_embeddings = model.encode([next_chunk_sents], normalize_embeddings=True)[0]
                sim_next = cosine_similarity([curr_chunk_embeddings], [next_chunk_embeddings])[0][0]
                options.append(('next', sim_next))
            if options:
                best_option = max(options, key=lambda x: x[1])
                if best_option[0] == 'prev':
                    chunks[i-1]['sents'].extend(chunk['sents'])
                    chunks[i-1]['vectors'].extend(chunk['vectors'])
                    del chunks[i]
                    i -= 1
                else:
                    chunks[i+1]['sents'] = chunk['sents'] + chunks[i+1]['sents']
                    chunks[i+1]['vectors'] = chunk['vectors'] + chunks[i+1]['vectors']
                    del chunks[i]
                    i -= 1
        i += 1
    return chunks

# Main Execution

In [ ]:
break_string = "================ {} ================"

input_csv = "../data/processed/all_articles.csv"
output_file = "../data/raw/segmented_data.parquet"

embedding_model_name = "intfloat/multilingual-e5-base"

z_threshold = -0.7
max_length_per_chunk = 300
min_syllable_count = 50
min_sentence_count = 2

In [ ]:
########### Load Data ###########
df = pd.read_csv(input_csv, encoding="utf-8")[['url', 'title', 'content']]
df = df.drop_duplicates(subset=['url']).reset_index(drop=True)
df = df.dropna(subset=['content']).reset_index(drop=True)

print(break_string.format("Dataset Overview"))
print(df.info())

########### Load Embedding Model ###########
print(break_string.format("Load Embedding Model"))
print("[INFO] Model name:", embedding_model_name)
start = time.perf_counter()
model = SentenceTransformer(embedding_model_name)
print("[INFO] Model's max sequence length:", model.max_seq_length)
end = time.perf_counter()
print(f"[INFO] Model loaded in {end - start:.2f} seconds.")

In [ ]:
########### Sentence Tokenization ###########
print(break_string.format("Sentence Tokenization"))
start = time.perf_counter()
df['sentences'] = df['content'].apply(sent_tokenize)
end = time.perf_counter()
print(f"[INFO] Sentence tokenization completed in {end - start:.2f} seconds.")

sentences_counts = df['sentences'].apply(len)
print(break_string.format("Sentence Count statistics"))
print(sentences_counts.describe())

########### Vectorization ###########
print(break_string.format("Vectorization"))
start = time.perf_counter()
all_embeddings = df['sentences'].apply(
    lambda sents: np.array(model.encode(sents, normalize_embeddings=True))
)
runtime = time.perf_counter() - start
print(f"[INFO] Vectorization completed in {runtime:.2f} seconds. ({runtime / len(df):.2f}s/sample)")

########### Adaptive Greedy Segmentation ###########
print(break_string.format("Greedy Adaptive Segmentation"))
start = time.perf_counter()
df['chunks'] = df.apply(
    lambda row: greedy_zscore_split(
        row['sentences'],
        all_embeddings.loc[row.name],
        z_threshold=z_threshold,
        max_length=max_length_per_chunk
    ),
    axis=1
)
runtime = time.perf_counter() - start
print(f"[INFO] Segmentation completed in {runtime:.2f} seconds. ({runtime/len(df):.2f}s/sample)")

print(break_string.format("Similarity-Based Chunk Merging"))
start = time.perf_counter()
df['chunks'] = df['chunks'].apply(
    lambda chunks: similarity_based_merge(
        chunks,
        model,
        min_syllable_count=min_syllable_count,
        min_sentence_count=min_sentence_count
    )
)
end = time.perf_counter()
print(f"[INFO] Chunk merging completed in {end - start:.2f} seconds.")

# Analysis

In [ ]:
# Statistics of: chunk counts per article, chunk sizes (in sentences and syllables)
print(break_string.format("Chunk Statistics"))
chunk_counts = df['chunks'].apply(len)
all_chunk_sentence_counts = []
all_chunk_syllable_counts = []
for chunks in df['chunks']:
    for chunk in chunks:
        all_chunk_sentence_counts.append(len(chunk['sents']))
        all_chunk_syllable_counts.append(get_approx_token_length(chunk))

print("Chunk counts per article:")
print(chunk_counts.describe())
print("\nChunk sizes (in sentences):")
print(pd.Series(all_chunk_sentence_counts).describe())
print("\nChunk sizes (in syllables):")
print(pd.Series(all_chunk_syllable_counts).describe())

# Save the segmented data

In [ ]:
# Convert chunks to text format (join sentences, remove vectors to save space)
df['chunks'] = df['chunks'].apply(
    lambda chunks_list: [' '.join(chunk['sents']) for chunk in chunks_list]
)

# Select columns and save to parquet (no explode - keeps list structure)
df_output = df[['url', 'title', 'chunks']]
df_output.to_parquet(output_file, index=False, engine='pyarrow', compression='snappy')

print(f"[INFO] Saved {len(df_output)} articles with total {sum(df_output['chunks'].apply(len))} chunks")
print(f"[INFO] File saved to: {output_file}")